# Per-Sensor Optimal $\tau$ — Empirical Analysis

**Two approaches compared:**

1. **Sweep**: for each sensor, scan $\tau$ and minimise the order-statistic NLL directly. No assumptions.
2. **Formula**: the pointwise-AMISE bandwidth $\tau^* = \bigl(g(t^*)/\bigl(n_{\rm eff}\,(g''(t^*))^2\bigr)\bigr)^{1/5}$, which assumes twice-differentiable density and MSE loss.

Run many events, collect per-sensor $(\tau^*_{\rm sweep},\, \text{features})$, see what predicts $\tau^*$, and check whether the formula agrees with the sweep.

In [ ]:
import sys
sys.path.append('..')

from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.generate import read_photon_data_from_photonsim
from lucid.detector_params import ParticleParams
from lucid.losses import first_arrival_nll, segment_logsumexp
from lucid.optimization.utils.functions import cartesian_to_spherical

import jax
import jax.numpy as jnp
import numpy as np
from jax import jit
from functools import partial
from scipy.special import logsumexp as scipy_logsumexp
from scipy.stats import spearmanr
import time

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

In [ ]:
default_json_filename = '../config/SK_geom_config.json'
PHYSICS_CONFIG = '../config/SK_physics_config.json'
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
TEMPERATURE = 0.10
K = 7
Nphot = 150_000

detector = generate_detector(default_json_filename)
NUM_DETECTORS = len(detector.all_points)

data_simulator = setup_event_simulator(
    default_json_filename, Nphot, temperature=0.0, K=20,
    is_data=True, is_calibration=False,
    physics_config=PHYSICS_CONFIG, default_detector_params=True)

prediction_simulator = setup_event_simulator(
    default_json_filename, Nphot, TEMPERATURE, max_sensors_per_cell=4, K=K,
    is_data=False, hit_mode='per_photon',
    physics_config=PHYSICS_CONFIG, default_detector_params=True)

@partial(jit, static_argnums=(5,))
def compute_sensor_nll(log_w, flat_times, flat_indices, t_obs, tau, num_detectors):
    return first_arrival_nll(log_w, flat_times, flat_indices, t_obs, tau, num_detectors)

# Wide sweep: 0.01 to 100, dense enough to resolve optima across 4 decades
N_TAU = 80
tau_grid = np.logspace(np.log10(0.01), np.log10(100.0), N_TAU)

print(f"Detectors: {NUM_DETECTORS}")
print(f"Tau grid: {N_TAU} points, [{tau_grid[0]:.3f}, {tau_grid[-1]:.1f}]")

In [ ]:
# Logistic kernel and its 2nd derivative
def logistic_K(u):
    s = 1.0 / (1.0 + np.exp(-np.clip(u, -500, 500)))
    return s * (1.0 - s)

def logistic_K2(u):
    s = 1.0 / (1.0 + np.exp(-np.clip(u, -500, 500)))
    K = s * (1.0 - s)
    return K * ((1.0 - 2.0*s)**2 - 2.0*K)


def process_event(entry_idx, rng_seed):
    # ---- load & pad photon data ----
    pd = read_photon_data_from_photonsim(data_file, entry_idx)
    N = len(pd['photon_origins'])
    pad = max(0, 1_000_000 - N)
    pd['photon_origins'] = jnp.pad(pd['photon_origins'], ((0,pad),(0,0)))
    d0 = jnp.array([0.,0.,1.])
    pd['photon_directions'] = (
        jnp.concatenate([pd['photon_directions'], jnp.tile(d0,(pad,1))]) if pad > 0
        else pd['photon_directions'])
    pd['photon_times'] = jnp.pad(pd['photon_times'], (0,pad))
    pd['N'] = N

    # ---- random track ----
    key = jax.random.PRNGKey(rng_seed)
    frac = 0.6
    r = jax.random.uniform(key, minval=0, maxval=detector.r*frac); key,_=jax.random.split(key)
    th = jax.random.uniform(key, minval=0, maxval=2*jnp.pi);      key,_=jax.random.split(key)
    z = jax.random.uniform(key, minval=-detector.H/2*frac, maxval=detector.H/2*frac); key,_=jax.random.split(key)
    pos = jnp.array([r*jnp.cos(th), r*jnp.sin(th), z])
    phi = jax.random.uniform(key, minval=0, maxval=2*jnp.pi); key,_=jax.random.split(key)
    ct = jax.random.uniform(key, minval=-1, maxval=1); key,_=jax.random.split(key)
    st = jnp.sqrt(1-ct**2)
    d = jnp.array([st*jnp.cos(phi), st*jnp.sin(phi), ct])

    # rotation
    o = jnp.array([0.,0.,1.])
    dn = d/(jnp.linalg.norm(d)+1e-8)
    ra = jnp.cross(o, dn); an = jnp.linalg.norm(ra)
    ra = jnp.where(an<1e-6, jnp.array([1.,0.,0.]), ra/(an+1e-8))
    rang = jnp.arccos(jnp.clip(jnp.dot(o, dn), -1., 1.))
    pd['rotation_axis']=ra; pd['rotation_angle']=rang
    pd['apply_rotation']=jnp.array(True)
    pd['apply_translation']=jnp.array(True)
    pd['translation_vector']=pos

    energy = pd['energy']
    theta_d, phi_d = cartesian_to_spherical(d)
    track = ParticleParams.from_cartesian(energy=energy, position=pos, direction=d, t0=0.)

    # ---- simulate ----
    key,_=jax.random.split(key)
    hc, ht = jax.lax.stop_gradient(data_simulator(track, key, pd))
    key,_=jax.random.split(key)
    tp = ParticleParams(energy=energy, position=pos, theta=theta_d, phi=phi_d, t0=jnp.array(0.))
    lw, ft, fi, tc = prediction_simulator(tp, key)

    # ---- tau sweep ----
    nll_mat = np.zeros((N_TAU, NUM_DETECTORS))
    for i, tau in enumerate(tau_grid):
        nll_mat[i] = np.array(compute_sensor_nll(lw, ft, fi, ht, tau, NUM_DETECTORS))

    hm = np.array(hc > 0)
    firing = np.where(hm)[0]
    toi = np.argmin(nll_mat[:, firing], axis=0)
    tau_opt = tau_grid[toi]

    # ---- per-sensor features (numpy) ----
    lw_np = np.array(lw); t_np = np.array(ft); idx_np = np.array(fi)
    tc_np = np.array(tc)
    v = lw_np > -20.0
    lw_v, t_v, idx_v = lw_np[v], t_np[v], idx_np[v]
    order = np.argsort(idx_v)
    lw_s, t_s, idx_s = lw_v[order], t_v[order], idx_v[order]
    uniq, cnts = np.unique(idx_s, return_counts=True)
    sp = np.concatenate([[0], np.cumsum(cnts)])
    s2i = {int(s):i for i,s in enumerate(uniq)}

    rows = []
    for j, s in enumerate(firing):
        if s not in s2i: continue
        si = s2i[s]; lo,hi = sp[si], sp[si+1]
        if hi-lo < 3: continue
        lw_g = lw_s[lo:hi]; t_g = t_s[lo:hi]
        to = np.argsort(t_g)
        ts = t_g[to]; lws = lw_g[to]

        Ns = float(np.exp(scipy_logsumexp(lws)))
        sw2 = float(np.exp(scipy_logsumexp(2*lws)))
        neff = Ns**2 / sw2 if sw2 > 0 else 1.0
        nraw = len(lws)

        log_p = lws - scipy_logsumexp(lws)
        p = np.exp(log_p)
        C = np.cumsum(p)
        target = min(1.0/max(Ns,1), 0.5)
        ks = min(int(np.searchsorted(C, target)), len(ts)-1)
        t_star = ts[ks]

        le = ts[ks] - ts[0]
        klo = max(0, ks-5); khi = min(len(ts)-1, ks+5)
        gaps = np.diff(ts[klo:khi+1]) if khi > klo else np.array([])
        lsp = float(np.median(gaps)) if len(gaps) > 0 else np.nan

        # pilot KDE for formula
        mu = np.sum(p * ts)
        sigma_w = np.sqrt(np.sum(p * (ts - mu)**2))
        h_pilot = max(sigma_w * neff**(-0.2), 1e-6)
        u_star = (t_star - ts) / h_pilot
        g_val = float(np.sum(p * logistic_K(u_star)) / h_pilot)
        gpp_val = float(np.sum(p * logistic_K2(u_star)) / h_pilot**3)

        if abs(gpp_val) > 1e-12 and g_val > 1e-12:
            tau_formula = float((g_val / (neff * gpp_val**2))**0.2)
        else:
            tau_formula = np.nan

        rows.append({
            'tau_sweep': tau_opt[j], 'N_s': Ns, 'n_raw': nraw,
            'n_eff': neff, 'local_spacing': lsp, 'leading_edge': le,
            'cdf_slope': g_val, 'g_pp': gpp_val, 'tau_formula': tau_formula,
        })
    return rows

print("Helpers defined")

In [ ]:
N_EVENTS = 50
all_rows = []

# warmup (first call triggers JIT)
_ = compute_sensor_nll(
    jnp.zeros(100), jnp.zeros(100), jnp.zeros(100, dtype=jnp.int32),
    jnp.zeros(NUM_DETECTORS), 0.1, NUM_DETECTORS)

t_start = time.time()
for ev in range(N_EVENTS):
    entry_idx = ev % 100
    rows = process_event(entry_idx, rng_seed=2000 + ev)
    all_rows.extend(rows)
    if (ev+1) % 10 == 0:
        elapsed = time.time() - t_start
        print(f"  Event {ev+1}/{N_EVENTS}  — {len(all_rows)} rows so far  ({elapsed:.0f}s)")

# Build arrays
data = {k: np.array([r[k] for r in all_rows]) for k in all_rows[0].keys()}
print(f"\nDone: {len(all_rows)} sensor-event rows from {N_EVENTS} events "
      f"in {time.time()-t_start:.0f}s")

In [ ]:
Ns = data['N_s']
ts = data['tau_sweep']

# N_s regime definitions
regime_low  = Ns < 1
regime_mid  = (Ns >= 1) & (Ns < 5)
regime_high = Ns >= 5

print("=== N_s regime breakdown ===")
for label, mask in [("N_s < 1", regime_low), ("1 <= N_s < 5", regime_mid), ("N_s >= 5", regime_high)]:
    n = np.sum(mask)
    at_lo = np.sum(ts[mask] == tau_grid[0])
    at_hi = np.sum(ts[mask] == tau_grid[-1])
    print(f"  {label:>14s}: {n:6d} sensors ({100*n/len(Ns):5.1f}%)  "
          f"tau at lower bound: {at_lo}  at upper bound: {at_hi}")

print(f"\n  Total: {len(Ns)}")
print(f"  Tau grid: [{tau_grid[0]:.3f}, {tau_grid[-1]:.1f}]")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# tau_sweep by regime
ax = axes[0]
for label, mask, color in [("$N_s<1$", regime_low, 'C0'),
                            ("$1{\\leq}N_s{<}5$", regime_mid, 'C1'),
                            ("$N_s{\\geq}5$", regime_high, 'C2')]:
    ax.hist(np.log10(ts[mask]), bins=60, alpha=0.5, label=label, color=color, edgecolor='black', lw=0.3)
ax.set_xlabel(r'$\log_{10}\,\tau^*_{\rm sweep}$'); ax.set_ylabel('count')
ax.set_title('Sweep-optimal tau by regime'); ax.legend(); ax.grid(alpha=0.3)

# N_s distribution
ax = axes[1]
ax.hist(np.log10(Ns), bins=60, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', ls='--', label='$N_s=1$')
ax.axvline(np.log10(5), color='orange', ls='--', label='$N_s=5$')
ax.set_xlabel(r'$\log_{10} N_s$'); ax.set_ylabel('count')
ax.set_title('Expected count distribution'); ax.legend(); ax.grid(alpha=0.3)

# tau vs N_s overview
ax = axes[2]
ax.scatter(np.log10(Ns), np.log10(ts), s=0.3, alpha=0.1, rasterized=True)
ax.axvline(0, color='red', ls='--', lw=1)
ax.axvline(np.log10(5), color='orange', ls='--', lw=1)
ax.set_xlabel(r'$\log_{10} N_s$'); ax.set_ylabel(r'$\log_{10} \tau^*$')
ax.set_title('Overview: all regimes'); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Log-log fits split by $N_s$ regime

Sensors with $N_s \ge 5$ have enough expected photons for the order-statistic model to be well-conditioned.
$1 \le N_s < 5$ is marginal. $N_s < 1$ means the sensor is expected to see less than one photon — the
first-arrival NLL is dominated by the survival term and $\tau$ has little to grab onto.

In [ ]:
features = [
    ('N_s',            r'$N_s$'),
    ('n_eff',          r'$n_{\rm eff}$'),
    ('n_raw',          r'$n_{\rm raw}$'),
    ('local_spacing',  'local spacing'),
    ('leading_edge',   'leading edge'),
    ('cdf_slope',      'CDF slope $g(t^*)$'),
]

regimes = [
    ("$N_s < 1$",              regime_low,  'C0'),
    ("$1 \\leq N_s < 5$",     regime_mid,  'C1'),
    ("$N_s \\geq 5$",         regime_high, 'C2'),
]

fit_results = {}  # key -> {regime_label: {slope, intercept, r2, rho, n}}

for key, label in features:
    fit_results[key] = {}
    x = data[key]
    for rlabel, rmask, _ in regimes:
        m = rmask & (x > 0) & (ts > 0) & np.isfinite(x) & np.isfinite(ts)
        if np.sum(m) < 10:
            fit_results[key][rlabel] = {'slope': np.nan, 'intercept': np.nan,
                                         'r2': np.nan, 'rho': np.nan, 'n': int(np.sum(m))}
            continue
        lx, lt = np.log(x[m]), np.log(ts[m])
        c = np.polyfit(lx, lt, 1)
        res = lt - np.polyval(c, lx)
        r2 = 1 - np.sum(res**2) / np.sum((lt - lt.mean())**2)
        rho, _ = spearmanr(lx, lt)
        fit_results[key][rlabel] = {'slope': c[0], 'intercept': c[1],
                                     'r2': r2, 'rho': rho, 'n': int(np.sum(m))}

# Plot: one row per feature, one column per regime
fig, axes = plt.subplots(len(features), 3, figsize=(15, 3.2*len(features)))

for row, (key, label) in enumerate(features):
    x = data[key]
    for col, (rlabel, rmask, color) in enumerate(regimes):
        ax = axes[row, col]
        m = rmask & (x > 0) & (ts > 0) & np.isfinite(x) & np.isfinite(ts)
        if np.sum(m) < 10:
            ax.text(0.5, 0.5, 'too few', transform=ax.transAxes, ha='center')
            continue
        lx, lt = np.log(x[m]), np.log(ts[m])
        ax.scatter(lx, lt, s=0.5, alpha=0.15, color=color, rasterized=True)
        fr = fit_results[key][rlabel]
        xf = np.linspace(np.percentile(lx, 2), np.percentile(lx, 98), 100)
        ax.plot(xf, fr['intercept'] + fr['slope']*xf, 'k-', lw=2)
        ax.set_title(f"{key}  {rlabel}\nslope={fr['slope']:.3f}  R²={fr['r2']:.3f}", fontsize=8)
        if col == 0: ax.set_ylabel(r'log $\tau^*$')
        if row == len(features)-1: ax.set_xlabel(f'log({key})')
        ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Summary table
print(f"\n{'feature':>15s}  {'regime':>22s}  {'n':>6s}  {'slope':>7s}  {'R2':>6s}  {'rho':>6s}")
print("-" * 75)
for key, _ in features:
    for rlabel, _, _ in regimes:
        fr = fit_results[key][rlabel]
        print(f"{key:>15s}  {rlabel:>22s}  {fr['n']:6d}  {fr['slope']:+7.3f}  {fr['r2']:6.3f}  {fr['rho']:+6.3f}")

## Formula vs sweep

If the AMISE formula $\tau^* = (g/n_{\rm eff}\,g''^2)^{1/5}$ reproduces the sweep result,
the Taylor-expansion assumptions hold at the Cherenkov leading edge.
If not, the sweep is ground truth and the formula should be discarded.

In [ ]:
m = (np.isfinite(data['tau_formula']) & (data['tau_formula'] > 0)
     & (data['tau_sweep'] > 0))
tf_all = data['tau_formula']
ts_all = data['tau_sweep']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Top row: formula vs sweep scatter per regime
for col, (rlabel, rmask, color) in enumerate(regimes):
    ax = axes[0, col]
    mm = m & rmask
    if np.sum(mm) < 10:
        ax.text(0.5, 0.5, 'too few', transform=ax.transAxes, ha='center')
        continue
    tf, tsw = tf_all[mm], ts_all[mm]
    ax.scatter(np.log(tsw), np.log(tf), s=0.5, alpha=0.15, color=color, rasterized=True)
    lims = [min(np.log(tsw).min(), np.log(tf).min()),
            max(np.log(tsw).max(), np.log(tf).max())]
    ax.plot(lims, lims, 'r--', lw=1)
    rho, _ = spearmanr(tsw, tf)
    ax.set_xlabel(r'$\log\tau^*_{\rm sweep}$')
    ax.set_ylabel(r'$\log\tau^*_{\rm formula}$')
    ax.set_title(f'{rlabel}  (n={np.sum(mm)}, rho={rho:.3f})', fontsize=9)
    ax.grid(alpha=0.3)

# Bottom row: log-ratio histograms per regime
for col, (rlabel, rmask, color) in enumerate(regimes):
    ax = axes[1, col]
    mm = m & rmask
    if np.sum(mm) < 10:
        ax.text(0.5, 0.5, 'too few', transform=ax.transAxes, ha='center')
        continue
    ratio = np.log(tf_all[mm] / ts_all[mm])
    ax.hist(ratio, bins=50, alpha=0.7, color=color, edgecolor='black', lw=0.3)
    ax.axvline(0, color='red', ls='--')
    ax.set_xlabel(r'$\log(\tau_{\rm formula}/\tau_{\rm sweep})$')
    ax.set_title(f'{rlabel}: mean={np.mean(ratio):.2f}, std={np.std(ratio):.2f}', fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Formula vs sweep by regime', y=1.01)
plt.tight_layout(); plt.show()

## Fit per regime

Fit $\log\tau = a + b\,\log N_s$ separately for each $N_s$ regime.
Compare R² within-regime vs the pooled fit to see if splitting helps.

In [ ]:
# Identify best predictor per regime from the fit_results table
print("Best single predictor per regime (by R²):")
print("-" * 65)
for rlabel, rmask, _ in regimes:
    best_k, best_r2 = None, -1
    for key, _ in features:
        fr = fit_results[key][rlabel]
        if fr['r2'] > best_r2:
            best_k, best_r2 = key, fr['r2']
    fr = fit_results[best_k][rlabel]
    print(f"  {rlabel:>22s}: {best_k:>15s}  slope={fr['slope']:+.3f}  R²={fr['r2']:.3f}")

# Per-regime fit using N_s (most interpretable, available from segment ops)
print("\n\nPer-regime N_s power-law fit:")
print("=" * 60)
regime_fits = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for col, (rlabel, rmask, color) in enumerate(regimes):
    ax = axes[col]
    m = rmask & (Ns > 0) & (ts > 0) & np.isfinite(Ns) & np.isfinite(ts)
    if np.sum(m) < 10:
        ax.text(0.5, 0.5, 'too few', transform=ax.transAxes, ha='center')
        continue
    lx, lt = np.log(Ns[m]), np.log(ts[m])
    c = np.polyfit(lx, lt, 1)
    res = lt - np.polyval(c, lx)
    r2 = 1 - np.sum(res**2) / np.sum((lt - lt.mean())**2)
    regime_fits[rlabel] = {'slope': c[0], 'intercept': c[1], 'r2': r2, 'n': int(np.sum(m))}

    ax.scatter(lx, lt, s=0.5, alpha=0.15, color=color, rasterized=True)
    xf = np.linspace(np.percentile(lx, 2), np.percentile(lx, 98), 100)
    ax.plot(xf, np.polyval(c, xf), 'k-', lw=2)
    ax.set_xlabel(r'$\log N_s$'); ax.set_ylabel(r'$\log \tau^*$')
    ax.set_title(f'{rlabel}\nslope={c[0]:.3f}  R²={r2:.3f}  n={np.sum(m)}', fontsize=9)
    ax.grid(alpha=0.3)

    print(f"  {rlabel:>22s}: tau ~ {np.exp(c[1]):.4f} * N_s^({c[0]:.3f})   R²={r2:.3f}   n={np.sum(m)}")

# Pooled fit for comparison
m_all = (Ns > 0) & (ts > 0) & np.isfinite(Ns) & np.isfinite(ts)
c_all = np.polyfit(np.log(Ns[m_all]), np.log(ts[m_all]), 1)
res_all = np.log(ts[m_all]) - np.polyval(c_all, np.log(Ns[m_all]))
r2_all = 1 - np.sum(res_all**2) / np.sum((np.log(ts[m_all]) - np.log(ts[m_all]).mean())**2)
print(f"\n  {'Pooled':>22s}: tau ~ {np.exp(c_all[1]):.4f} * N_s^({c_all[0]:.3f})   R²={r2_all:.3f}   n={np.sum(m_all)}")

plt.tight_layout(); plt.show()

## Should $N_s < 1$ sensors be in the timing loss?

For sensors expecting less than one photon, the order-statistic NLL
$-\log N_s - \log f - (N_s{-}1)\log(1{-}F)$ has $N_s < 1$, so the
survival term $(N_s{-}1)\log(1{-}F)$ flips sign. The NLL becomes nearly
flat in $\tau$ — these sensors contribute noise, not signal.

Check: how sensitive is the NLL to $\tau$ in each regime?
If the NLL curve is flat for $N_s < 1$, these sensors can be dropped.

In [ ]:
# For each regime, measure how much the NLL varies across the tau sweep.
# NLL range = max(NLL) - min(NLL) over tau, per sensor.
# If it's small, tau doesn't matter and the sensor adds no timing information.

# Re-run one event to get full NLL curves
print("Re-running one event to get full NLL(tau) curves...")
test_rows = process_event(0, rng_seed=9999)
test_Ns = np.array([r['N_s'] for r in test_rows])

# We need the actual nll_matrix — extract it by re-doing the sweep inline
pd_test = read_photon_data_from_photonsim(data_file, 0)
N_test = len(pd_test['photon_origins'])
pad_test = max(0, 1_000_000 - N_test)
pd_test['photon_origins'] = jnp.pad(pd_test['photon_origins'], ((0,pad_test),(0,0)))
d0 = jnp.array([0.,0.,1.])
pd_test['photon_directions'] = (
    jnp.concatenate([pd_test['photon_directions'], jnp.tile(d0,(pad_test,1))]) if pad_test > 0
    else pd_test['photon_directions'])
pd_test['photon_times'] = jnp.pad(pd_test['photon_times'], (0,pad_test))
pd_test['N'] = N_test

key_t = jax.random.PRNGKey(9999)
frac = 0.6
r_t = jax.random.uniform(key_t, minval=0, maxval=detector.r*frac); key_t,_=jax.random.split(key_t)
th_t = jax.random.uniform(key_t, minval=0, maxval=2*jnp.pi); key_t,_=jax.random.split(key_t)
z_t = jax.random.uniform(key_t, minval=-detector.H/2*frac, maxval=detector.H/2*frac); key_t,_=jax.random.split(key_t)
pos_t = jnp.array([r_t*jnp.cos(th_t), r_t*jnp.sin(th_t), z_t])
phi_t = jax.random.uniform(key_t, minval=0, maxval=2*jnp.pi); key_t,_=jax.random.split(key_t)
ct_t = jax.random.uniform(key_t, minval=-1, maxval=1); key_t,_=jax.random.split(key_t)
st_t = jnp.sqrt(1-ct_t**2)
d_t = jnp.array([st_t*jnp.cos(phi_t), st_t*jnp.sin(phi_t), ct_t])

o = jnp.array([0.,0.,1.])
dn_t = d_t/(jnp.linalg.norm(d_t)+1e-8)
ra_t = jnp.cross(o, dn_t); an_t = jnp.linalg.norm(ra_t)
ra_t = jnp.where(an_t<1e-6, jnp.array([1.,0.,0.]), ra_t/(an_t+1e-8))
rang_t = jnp.arccos(jnp.clip(jnp.dot(o, dn_t), -1., 1.))
pd_test['rotation_axis']=ra_t; pd_test['rotation_angle']=rang_t
pd_test['apply_rotation']=jnp.array(True); pd_test['apply_translation']=jnp.array(True)
pd_test['translation_vector']=pos_t

energy_t = pd_test['energy']
theta_dt, phi_dt = cartesian_to_spherical(d_t)
track_t = ParticleParams.from_cartesian(energy=energy_t, position=pos_t, direction=d_t, t0=0.)
key_t,_=jax.random.split(key_t)
hc_t, ht_t = jax.lax.stop_gradient(data_simulator(track_t, key_t, pd_test))
key_t,_=jax.random.split(key_t)
tp_t = ParticleParams(energy=energy_t, position=pos_t, theta=theta_dt, phi=phi_dt, t0=jnp.array(0.))
lw_t, ft_t, fi_t, tc_t = prediction_simulator(tp_t, key_t)

nll_mat_t = np.zeros((N_TAU, NUM_DETECTORS))
for i, tau in enumerate(tau_grid):
    nll_mat_t[i] = np.array(compute_sensor_nll(lw_t, ft_t, fi_t, ht_t, tau, NUM_DETECTORS))

hm_t = np.array(hc_t > 0)
firing_t = np.where(hm_t)[0]
Ns_t = np.array(tc_t)[firing_t]

# NLL range per sensor
nll_range = np.ptp(nll_mat_t[:, firing_t], axis=0)  # max - min over tau

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# NLL range vs N_s
ax = axes[0]
ax.scatter(np.log10(Ns_t), nll_range, s=1, alpha=0.3, rasterized=True)
ax.axvline(0, color='red', ls='--', label='$N_s=1$')
ax.axvline(np.log10(5), color='orange', ls='--', label='$N_s=5$')
ax.set_xlabel(r'$\log_{10} N_s$'); ax.set_ylabel('NLL range over tau sweep')
ax.set_title('NLL sensitivity to tau'); ax.legend(); ax.grid(alpha=0.3)

# Box plot by regime
ax = axes[1]
groups = []
labels_box = []
for rlabel, lo, hi in [('$N_s<1$', 0, 1), ('$1{\\leq}N_s{<}5$', 1, 5), ('$N_s{\\geq}5$', 5, 1e10)]:
    mask_r = (Ns_t >= lo) & (Ns_t < hi)
    if np.sum(mask_r) > 0:
        groups.append(nll_range[mask_r])
        labels_box.append(rlabel)
bp = ax.boxplot(groups, labels=labels_box, patch_artist=True)
for patch, c in zip(bp['boxes'], ['C0','C1','C2']): patch.set_facecolor(c); patch.set_alpha(0.5)
ax.set_ylabel('NLL range'); ax.set_title('NLL range by regime'); ax.grid(alpha=0.3)

# Example NLL curves: one sensor per regime
ax = axes[2]
for lo, hi, color, rlabel in [(0,1,'C0','$N_s<1$'), (1,5,'C1','$1{\\leq}N_s{<}5$'), (5,1e10,'C2','$N_s{\\geq}5$')]:
    mask_r = (Ns_t >= lo) & (Ns_t < hi)
    if np.sum(mask_r) == 0: continue
    # pick the sensor closest to median N_s in this regime
    Ns_r = Ns_t[mask_r]
    idx_r = np.where(mask_r)[0]
    med_idx = idx_r[np.argmin(np.abs(Ns_r - np.median(Ns_r)))]
    s = firing_t[med_idx]
    curve = nll_mat_t[:, s]
    curve_shifted = curve - curve.min()  # shift to 0 for comparison
    ax.plot(tau_grid, curve_shifted, color=color, lw=1.5,
            label=f'{rlabel} ($N_s$={Ns_t[med_idx]:.2f})')
ax.set_xscale('log'); ax.set_xlabel(r'$\tau$'); ax.set_ylabel('NLL - min(NLL)')
ax.set_title('Example NLL curves (shifted)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Summary statistics
print("\nNLL sensitivity by regime:")
print(f"{'Regime':>22s}  {'n':>6s}  {'median range':>12s}  {'mean range':>11s}  {'90th pctile':>11s}")
for rlabel, lo, hi in [('N_s < 1', 0, 1), ('1 <= N_s < 5', 1, 5), ('N_s >= 5', 5, 1e10)]:
    mask_r = (Ns_t >= lo) & (Ns_t < hi)
    if np.sum(mask_r) == 0: continue
    r = nll_range[mask_r]
    print(f"{rlabel:>22s}  {np.sum(mask_r):6d}  {np.median(r):12.3f}  {np.mean(r):11.3f}  {np.percentile(r,90):11.3f}")

## Summary

**Three questions answered:**

1. **Does the sweep range matter?** Yes — the previous upper bound of 5.0 was capping low-$N_s$ sensors.
   Widening to 100 gives uncontaminated optima.

2. **Does splitting by $N_s$ regime help?** Check the per-regime R² values above vs the pooled fit.
   If the within-regime slopes differ substantially, the relationship is not a single power law.

3. **Should $N_s < 1$ sensors contribute to the timing loss?** Look at the NLL range plot. If the
   NLL is nearly flat across the entire $\tau$ sweep, these sensors carry no timing information and
   are adding noise to the gradient. Consider masking them out with `hit_mask = (total_charge > 1)`.

**Deliverable**: a per-regime formula $\tau_s = c \cdot N_s^\alpha$ (or whichever predictor wins),
applied only to sensors above the $N_s$ threshold where $\tau$ matters.